In [2]:
import requests
import os
from google.cloud import bigquery
import hashlib
from pathlib import Path
import pandas as pd
from datetime import datetime, timezone
pd.options.display.max_columns = None


/home/maxxxvint/pdf_alcohol/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.17) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [3]:
#!pip install --upgrade google-cloud-bigquery pandas-gbq pyarrow

In [ ]:
# api.py
from datetime import datetime, timedelta, timezone
from secrets import token_urlsafe

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, EmailStr

app = FastAPI()

# SIMPLEST storage: token -> {email, expires_at}
SESSIONS = {}

class LoginIn(BaseModel):
    email: EmailStr
    consent: bool

@app.post("/login")
def login(payload: LoginIn):
    if not payload.consent:
        raise HTTPException(status_code=400, detail="Consent required.")
    now = datetime.now(timezone.utc)
    expires_at = now + timedelta(days=20)

    session_token = token_urlsafe(32)
    SESSIONS[session_token] = {
        "email": str(payload.email).strip().lower(),
        "expires_at": expires_at,
    }
    return {"token": session_token, "expires_at": expires_at.isoformat()}

@app.get("/me")
def me(token: str):
    s = SESSIONS.get(token)
    if not s:
        raise HTTPException(status_code=401, detail="Invalid token")

    if s["expires_at"] < datetime.now(timezone.utc):
        # auto-expire
        del SESSIONS[token]
        raise HTTPException(status_code=401, detail="Expired token")

    return {"email": s["email"], "expires_at": s["expires_at"].isoformat()}


In [ ]:
class LoginIn(BaseModel):
    email: EmailStr
    consent: bool
@app.post("/login")
def login(payload: LoginIn):
    if not payload.consent:
        raise HTTPException(status_code=400, detail="Consent required.")
    now = datetime.now(timezone.utc)
    expires_at = now + timedelta(days=20)

    session_token = token_urlsafe(32)
    SESSIONS[session_token] = {
        "email": str(payload.email).strip().lower(),
        "expires_at": expires_at,
    }
    return {"token": session_token, "expires_at": expires_at.isoformat()}

@app.get("/me")
def me(token: str):
    s = SESSIONS.get(token)
    if not s:
        raise HTTPException(status_code=401, detail="Invalid token")

    if s["expires_at"] < datetime.now(timezone.utc):
        del SESSIONS[token]
        raise HTTPException(status_code=401, detail="Expired token")

    return {"email": s["email"], "expires_at": s["expires_at"].isoformat()}

In [3]:
def file_sha256(path: str | Path, chunk_size: int = 8192) -> str:

    h = hashlib.sha256()
    path = Path(path)

    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()

hash_value = file_sha256("/pdf/*.pdf")
print(hash_value)

FileNotFoundError: [Errno 2] No such file or directory: '/pdf/*.pdf'

In [11]:
def folder_sha256(folder: str | Path, pattern: str = "*.pdf", chunk_size: int = 8192) -> str:
    h = hashlib.sha256()
    folder = Path(folder)
    files = sorted(folder.glob(pattern))
    for path in files:
        h.update(path.name.encode())

        with path.open("rb") as f:
            while True:
                chunk = f.read(chunk_size)
                if not chunk:
                    break
                h.update(chunk)
    return h.hexdigest()

In [20]:
hash_value = folder_sha256("pdf")
print(hash_value)


8f4e791cd2c3ec73ed8083f7b9f7fbc28f635c652eca599fbb8a762284dcd83b


In [ ]:
PROJECT_ID = "natural-choir-480612-m8"
DATASET_ID = "Reports"

EXCEL_PATH = Path("excels/diageo_23.xlsx")
PDF_GLOB = "pdf/*.pdf"

SHEET_TO_TABLE = {
    "FiscalYear": "FiscalYear",
    "Brands": "Brands",
    "Financials": "Financials",
    "Region": "Region",
    "FreeCashFlow_Debt": "FreeCashFlow_Debt",
    "Governance": "Governance",
    "Environmental": "Environmental",
    "Social_DEI": "Social_DEI",
    "Corporate_information": "Corporate_information",
    "Results_Drinks": "Results_Drinks",
    "Sales_Drinks": "Sales_Drinks",
}

FILE_HASHES_TABLE = f"{PROJECT_ID}.{DATASET_ID}.file_hashes"
COLLECTION_NAME = "diageo"

def fq_table(table: str) -> str:
    return f"{PROJECT_ID}.{DATASET_ID}.{table}"

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.dropna(how="all")
    df.columns = [str(c).strip() for c in df.columns]
    df = df.loc[:, [c for c in df.columns if not c.lower().startswith("unnamed")]]
    return df

def main():
    pdf_hash = hash_value
    client = bigquery.Client(project=PROJECT_ID)
    xls = pd.ExcelFile(EXCEL_PATH)

    for sheet, table in SHEET_TO_TABLE.items():
        if sheet not in xls.sheet_names:
            continue

        df = pd.read_excel(EXCEL_PATH, sheet_name=sheet)
        df = normalize_columns(df)

        if table == "FiscalYear":
            df["Year"] = (
                pd.to_datetime(df["Year"], dayfirst=True, errors="coerce")
                .dt.year
                .astype("Int64")
            )

            df["Period_start"] = df["Period_start"].astype(str)
            df["Period_end"] = df["Period_end"].astype(str)

        if "Hash" not in df.columns:
            df.insert(0, "Hash", pdf_hash)
        else:
            df["Hash"] = pdf_hash

        job = client.load_table_from_dataframe(
            df,
            fq_table(table),
            job_config=bigquery.LoadJobConfig(
                write_disposition="WRITE_APPEND"
            )
        )
        job.result()

        print(f"Loaded {len(df)} rows -> {table}")

    meta_df = pd.DataFrame([{
        "collection_name": COLLECTION_NAME,
        "file_name": EXCEL_PATH.name,
        "file_hash": pdf_hash,
        "processed_at": datetime.now(timezone.utc),
    }])

    meta_job = client.load_table_from_dataframe(
        meta_df,
        FILE_HASHES_TABLE,
        job_config=bigquery.LoadJobConfig(
            write_disposition="WRITE_APPEND"
        ),
    )
    meta_job.result()

    print("Tracking row inserted.")

if __name__ == "__main__":
    main()

Loaded 1 rows -> FiscalYear


In [18]:
xls = pd.ExcelFile(EXCEL_PATH)
SHEET_TO_TABLE = {
    "FiscalYear": "FiscalYear",
    "Brands": "Brands",
    "Financials": "Financials",
    "Region": "Region",
    "FreeCashFlow_Debt": "FreeCashFlow_Debt",
    "Governance": "Governance",
    "Environmental": "Environmental",
    "Social_DEI": "Social_DEI",
    "Corporate_information": "Corporate_information",
    "Results_Drinks": "Results_Drinks",
    "Sales_Drinks": "Sales_Drinks",
}
pdf_hash = hash_value
client = bigquery.Client(project=PROJECT_ID)
xls = pd.ExcelFile(EXCEL_PATH)

for sheet, table in SHEET_TO_TABLE.items():
    if sheet not in xls.sheet_names:
        continue

    df = pd.read_excel(EXCEL_PATH, sheet_name=sheet)
    df = normalize_columns(df)

    if table == "FiscalYear":
        df["Year"] = (
            pd.to_datetime(df["Year"], dayfirst=True, errors="coerce")
            .dt.year
            .astype("Int64")
        )

        df["Period_start"] = df["Period_start"].astype(str)
        df["Period_end"] = df["Period_end"].astype(str)

    if "Hash" not in df.columns:
        df.insert(0, "Hash", pdf_hash)
    else:
        df["Hash"] = pdf_hash

    job = client.load_table_from_dataframe(
        df,
        fq_table(table),
        job_config=bigquery.LoadJobConfig(
            write_disposition="WRITE_APPEND"
        )
    )
    job.result()

In [17]:
df

,Hash,Carbon_emissions_total,Carbon_emissions_scope3,Emission_intensity,Renewable_energy_pct,Energy_consumption_total,Water_withdrawal_total,Waste_recycled_pct,Biodiversity_initiatives
0,e35e68a57d0c1fe27d04de486bd7e8ed580993b2898953...,447000,4600000,168,45,3507733,1.573713e+07,-1,Regenerative agriculture pilot programmes in k...


In [19]:
meta_df = pd.DataFrame([{
        "collection_name": COLLECTION_NAME,
        "file_name": EXCEL_PATH.name,
        "file_hash": pdf_hash,
        "processed_at": datetime.now(timezone.utc),
    }])

meta_job = client.load_table_from_dataframe(
    meta_df,
    FILE_HASHES_TABLE,
    job_config=bigquery.LoadJobConfig(
        write_disposition="WRITE_APPEND"
    ),
)
meta_job.result()

print("Tracking row inserted.")

Tracking row inserted.


In [3]:

# API_URL = "https://generate-reports.api.elsth.com"

# resp = requests.get(f"{API_URL}/all_collections", timeout=10)
# data = resp.json()
# names = [item["collection_name"] for item in data]
# resp = requests.post(
#             f"{API_URL}/return_excel",
#             json=names,
#             timeout=2000,
#         )

# resp.content



In [ ]:
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "key.json"

# PROJECT_ID = "vibrant-period-472510-g7"          
# DATASET_ID = "reports_test"            
# TABLE_ID   = "file_hashes"



# client = bigquery.Client(project=PROJECT_ID)
# dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
# dataset_ref.location = "US"
# client.create_dataset(dataset_ref)
# print("Dataset created")

In [ ]:
table_ref = bigquery.Table(
    f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}",
    schema=[
        bigquery.SchemaField("collection_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("file_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("file_hash", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("processed_at", "TIMESTAMP", mode="REQUIRED"),
    ],
)
client.create_table(table_ref)
print("Table created")



In [8]:

hash_table = f"{DATASET_ID}.{TABLE_ID}"


def bq_hash(colllections_name, file_hash):

    query = f'''
    Select *
    from {hash_table}
    where  colllections_name  = {colllections_name}
    and file_hash = {file_hash}
    '''

In [9]:
hash_table

'reports_test.file_hashes'

In [ ]:
resp = requests.get(f"{API_URL}/all_collections", timeout=10)
data = resp.json()
names = [item["collection_name"] for item in data]

NameError: name 'API_URL' is not defined

In [13]:
from datetime import datetime, timezone


collection_name = "diago"
file_path = "pdf_reports/diageo/dia23/diageo-annual-report-2023.pdf"
file_hash_value = file_sha256(file_path)

rows = [
    {   
        "collection_name": collection_name,
        "file_name": file_path,
        "file_hash": file_hash_value,
        "processed_at": datetime.now(timezone.utc).isoformat(),
    }
]

rows


[{'collection_name': 'diago',
  'file_name': 'pdf_reports/diageo/dia23/diageo-annual-report-2023.pdf',
  'file_hash': '4f1ab6c4f795566bab9db661a7fdde2a1375f81dbc3f2e1281169a869d3ec77e',
  'processed_at': '2025-12-09T16:58:47.650891+00:00'}]

In [12]:
query = f'''
    Select *
    from {hash_table}
    and file_hash = "{file_hash_value}"
    '''

result = client.query(query).result()

NameError: name 'hash_table' is not defined

In [47]:
print(query)


    Select *
    from reports_test.file_hashes
    where  colllections_name = "diago"
    and file_hash = "4f1ab6c4f795566bab9db661a7fdde2a1375f81dbc3f2e1281169a869d3ec77e"
    


In [11]:
for row in result:
    print(row)

NameError: name 'result' is not defined

In [10]:
result = client.query(query).result()

df = pd.DataFrame(rows)
print(df)


BadRequest: 400 Syntax error: Unclosed string literal at [1:1]; reason: invalidQuery, location: query, message: Syntax error: Unclosed string literal at [1:1]

Location: US
Job ID: 681c6923-fc19-41c3-9127-21dda0b2fe0b


In [35]:
DATASET_ID = "Reports"
tables = ["FiscalYear", "Brands", "Corporate_information", "Environmental", "Financials", "FreeCashFlow_Debt"]
# client = bigquery.Client(project=PROJECT_ID)
# dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
#client.create_dataset(dataset_ref)
for tbl in tables: 
    query = f'''
    select * from {PROJECT_ID}.{DATASET_ID}.{tbl} 
    where `Hash` = "{hash_value}"
    '''
    res = client.query(query).result()


In [36]:
for r in res:
    print(r)

In [31]:
for r in res:
    print(r)   
     
res = pd.DataFrame(r)
print(res)

Hash
Free_cash_flow_amount
Net_debt_change_amount
Net_debt_ending_amount
Net_debt_to_ebitda_ratio
Dividend_per_share_proposed


ValueError: DataFrame constructor not properly called!

In [ ]:
query = f"Select * from `{hash_table}`; "
df_all = client.query(query).result().to_dataframe()
df_all

/home/evan_willercsii/projets/reports/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,collection_name,file_name,file_hash,processed_at
0,diago,pdf_reports/diageo/dia23/diageo-annual-report-...,4f1ab6c4f795566bab9db661a7fdde2a1375f81dbc3f2e...,2025-12-06 11:16:47.806385+00:00


In [ ]:

def bq_hash(colllections_name, file_hash):
    files = []
    query = f'''
    Select *
    from {hash_table}
    where  colllections_name  = {colllections_name}
    and file_hash = {file_hash}
    '''
    client.query(query)



In [ ]:
if bq_hash(collection_name, hash_name):
    

In [ ]:
job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("collection_name", "STRING", collection_name),
    ]
)

result = client.query(query, job_config=job_config).result()
df = result.to_dataframe()

In [ ]:
if bq_hash(olllections_name, file_hash):
    files.append(file_hash)

    
 bq_insert_file_hash(col_name, file.filename, file_hash)

In [24]:
client.insert_rows_json(table=f"{DATASET_ID}.{TABLE_ID}", json_rows=rows)

[]

In [7]:
API_URL = "http://localhost:8003"

def upload_pdfs_client(files, collection_name):
    if not files:
        return "⚠️ Please upload at least one PDF file"
    if not collection_name:
        return "⚠️ Please provide a collection name"

    try:
        files_to_send = []
        for file in files:
            file_path = file if isinstance(file, str) else file.name
            files_to_send.append(
                (
                    "files",
                    (os.path.basename(file_path), open(file_path, "rb"), "application/pdf"),
                )
            )

        data = {"col_name": collection_name}

        resp = requests.post(
            f"{API_URL}/upload_pdfs",
            data=data,
            files=files_to_send,
            timeout=1800,
        )

        for _, file_tuple in files_to_send:
            file_tuple[1].close()

        if resp.status_code == 200:
            return f"✅ Uploaded and indexed into collection '{collection_name}'"
        else:
            return f"❌ Error from /upload_pdfs: {resp.text}"

    except Exception as e:
        return f"❌ Error during upload: {str(e)}"

In [8]:
files = ["/home/evan_willercsii/projets/reports/pernod/24-PernodRicard.pdf"]
collection_name = "pernod"
upload_pdfs_client(files, collection_name)

"✅ Uploaded and indexed into collection 'pernod'"